# T2 · Declare the Grain
Parts A, B and C below are subtasks of this one task, so they live in this one notebook.

## Part A · PakFreight grain statement

In [1]:
import re, pandas as pd
pd.set_option("display.max_colwidth", None, "display.width", 250)

def grain_ok(s):
    """Grain mark is lost if the sentence contains the word 'and' (also guard against the substring)."""
    return re.search(r"\band\b", s, re.I) is None and "and" not in s.lower()

grain_A = ("One row of FactShipment represents one shipment leg: a single consignment moved by one mode "
           "from one origin city to one destination city on one ship date.")
print(grain_A)
print("Contains 'and'?", not grain_ok(grain_A))
assert grain_ok(grain_A)

One row of FactShipment represents one shipment leg: a single consignment moved by one mode from one origin city to one destination city on one ship date.
Contains 'and'? False


**Fact table type:** transaction. **Why leg, not shipment:** if one consignment is later carried marine-then-road, each carried segment stays one row; a "shipment" row would become ambiguous. `DimCity` plays two roles (origin, destination).

## Part B · Four new processes

In [2]:
B = [
 (1, "Fuel purchases at PakFreight depots",
     "One row per fuel purchase: one fuelling receipt for one truck at one depot on one date.",
     "DimDate, DimDepot (DimCity role), DimTruck, DimDriver, DimFuelType, DimSupplier; ReceiptNumber as degenerate dimension",
     "Litres (additive), FuelCostPKR (additive); price per litre is not stored",
     "Transaction: each purchase is an event that happens once and is never revised. One litre is too fine (no extra information), one truck-week is an aggregate that discards the purchase."),
 (2, "Driver duty hours and rest compliance",
     "One row per completed duty shift: one driver, from clock-on to clock-off, on one start date.",
     "DimDate (start date), DimDriver, DimDepot, DimShiftType, DimTruck",
     "DutyHours (additive), RestHoursSincePrevShift (additive within a driver)",
     "Transaction. The rolling 14-day window changes the measures, not the grain: it is computed over DimDate by summing shift rows, never stored."),
 (3, "Customer complaints about late delivery",
     "One row per complaint lodged by a customer about late delivery, updated as it moves from acknowledgement to resolution.",
     "DimCustomer, DimDate (three roles: filed, acknowledged, resolved), DimComplaintStatus, DimComplaintCategory, DimChannel + BridgeComplaintShipment (not a dimension)",
     "ComplaintCount = 1 (additive), DaysToResolve (additive; NULL until resolved, divide by ResolvedCount for an average)",
     "Accumulating snapshot: one row has a lifecycle (filed, acknowledged, resolved) and the row is updated as milestones are reached. One complaint can cite several shipments, so the many-to-many lives in a bridge table with an AllocationWeight that sums to 1.0 per complaint."),
 (4, "Daily warehouse stock levels at Lahore Dry Port",
     "One row per cargo class per day: the closing stock of one cargo class in the Lahore Dry Port warehouse on one calendar date.",
     "DimDate, DimWarehouse, DimCargoClass",
     "ClosingStockKG (semi-additive), PalletSlotsOccupied (semi-additive); utilisation % is derived from occupied slots plus slot capacity, never stored",
     "Periodic snapshot: stock is a state, not an event, so a row is written every day including quiet days. Never SUM across days; use average or last value."),
]
t2b = pd.DataFrame(B, columns=["#", "Process", "Grain (one row per ...)", "Dimensions", "Two measures", "Fact type + why"])
assert all(grain_ok(g) for g in t2b["Grain (one row per ...)"]), "a grain sentence contains 'and'"
t2b[["#", "Process", "Grain (one row per ...)", "Fact type + why"]]

,#,Process,Grain (one row per ...),Fact type + why
0,1,Fuel purchases at PakFreight depots,One row per fuel purchase: one fuelling receipt for one truck at one depot on one date.,"Transaction: each purchase is an event that happens once and is never revised. One litre is too fine (no extra information), one truck-week is an aggregate that discards the purchase."
1,2,Driver duty hours and rest compliance,"One row per completed duty shift: one driver, from clock-on to clock-off, on one start date.","Transaction. The rolling 14-day window changes the measures, not the grain: it is computed over DimDate by summing shift rows, never stored."
2,3,Customer complaints about late delivery,"One row per complaint lodged by a customer about late delivery, updated as it moves from acknowledgement to resolution.","Accumulating snapshot: one row has a lifecycle (filed, acknowledged, resolved) and the row is updated as milestones are reached. One complaint can cite several shipments, so the many-to-many lives in a bridge table with an AllocationWeight that sums to 1.0 per complaint."
3,4,Daily warehouse stock levels at Lahore Dry Port,One row per cargo class per day: the closing stock of one cargo class in the Lahore Dry Port warehouse on one calendar date.,"Periodic snapshot: stock is a state, not an event, so a row is written every day including quiet days. Never SUM across days; use average or last value."


In [3]:
t2b[["#", "Dimensions", "Two measures"]]

,#,Dimensions,Two measures
0,1,"DimDate, DimDepot (DimCity role), DimTruck, DimDriver, DimFuelType, DimSupplier; ReceiptNumber as degenerate dimension","Litres (additive), FuelCostPKR (additive); price per litre is not stored"
1,2,"DimDate (start date), DimDriver, DimDepot, DimShiftType, DimTruck","DutyHours (additive), RestHoursSincePrevShift (additive within a driver)"
2,3,"DimCustomer, DimDate (three roles: filed, acknowledged, resolved), DimComplaintStatus, DimComplaintCategory, DimChannel + BridgeComplaintShipment (not a dimension)","ComplaintCount = 1 (additive), DaysToResolve (additive; NULL until resolved, divide by ResolvedCount for an average)"
3,4,"DimDate, DimWarehouse, DimCargoClass","ClosingStockKG (semi-additive), PalletSlotsOccupied (semi-additive); utilisation % is derived from occupied slots plus slot capacity, never stored"


**Bonus:** #3 is not a plain transaction table (accumulating snapshot) and needs a bridge table, which is not a dimension; #4 is a periodic snapshot.

## Part C · The grain that went wrong

In [4]:
import os
p = next(x for x in ["Shipments_Unified.csv", "../Shipments_Unified.csv", "/mnt/user-data/uploads/Shipments_Unified.csv"] if os.path.exists(x))
src = pd.read_csv(p)

bad = pd.concat([src, src[src.ShipmentID == "RD-501"]], ignore_index=True)   # row duplicated for the 2nd customer
checks = pd.DataFrame({
    "T7 check": ["#1 Row count", "#2 Total WeightKG", "#3 Total FreightCostPKR", "#4 Shipment legs (SUM of Count)", "#5 Distinct ShipmentID"],
    "Expected": [34, 578_650, 4_808_000, 34, 34],
    "After duplicate": [len(bad), bad.WeightKG.sum(), bad.CostPKR.sum(), len(bad), bad.ShipmentID.nunique()],
})
checks["Pass?"] = (checks.Expected == checks["After duplicate"]).map({True: "pass", False: "FAIL"})
checks

,T7 check,Expected,After duplicate,Pass?
0,#1 Row count,34,35,FAIL
1,#2 Total WeightKG,578650,585150,FAIL
2,#3 Total FreightCostPKR,4808000,4866000,FAIL
3,#4 Shipment legs (SUM of Count),34,35,FAIL
4,#5 Distinct ShipmentID,34,34,pass


In [5]:
# Correct design: fact stays at 34 rows; customers hang off a bridge with allocation weights summing to 1.0
# (customers and weights below are illustrative - RD-501 is the shipment from the case, the split is invented)
bridge = pd.DataFrame({"BillingGroupKey": [1, 1], "CustomerKey": [101, 102], "AllocationWeight": [0.5, 0.5]})
assert bridge.groupby("BillingGroupKey").AllocationWeight.sum().eq(1.0).all()

w = src.loc[src.ShipmentID == "RD-501", "WeightKG"].iloc[0]
alloc = bridge.assign(WeightKG=w * bridge.AllocationWeight)
assert alloc.WeightKG.sum() == w and len(src) == 34 and src.WeightKG.sum() == 578_650
alloc

,BillingGroupKey,CustomerKey,AllocationWeight,WeightKG
0,1,101,0.5,3250.0
1,1,102,0.5,3250.0


**(a) Rule broken:** the declared grain (one row per shipment leg). Duplicating the row made it "one row per leg per customer" for one leg only, so the table now has two grains; and a descriptive column (`CustomerName`) went into the fact. Totals moved by 6,500 kg.

**(b) Should have built:** `DimCustomer` plus a `BillingGroupKey` in the fact pointing to a bridge table (`BillingGroupKey`, `CustomerKey`, `AllocationWeight` summing to 1.0 per group). The fact stays at 34 rows and the weighted split keeps the totals.

**(c) Caught by:** T7 check **#1** (row count 35, expected 34), and also **#2** (585,150 kg), **#3** (cost 4,866,000) and **#4** (leg count 35). Check #5 would not fire: distinct ShipmentID stays 34.